In [ ]:
!pip install transformers tqdm torch numpy pandas lmdb

from transformers import AutoModelForMaskedLM

import pickle
import io
import lmdb
import os
import tqdm
import torch
import numpy as np
import pandas as pd
import gc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 6.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/Github/Mprotein_hydrophobic/'
os.makedirs(SAVE_PATH, exist_ok=True)
df_raw = pd.read_csv('/content/drive/MyDrive/Github/Mprotein_hydrophobic/dataset_.csv')
df = df_raw.copy()

Mounted at /content/drive


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = 'Synthyra/ESMplusplus_large'
MODEL_PRETRAINED = AutoModelForMaskedLM.from_pretrained(MODEL_ID, trust_remote_code=True).to(DEVICE).eval()
TOKENIZER = MODEL_PRETRAINED.tokenizer
Y_LABELS = ['Peripheral', 'Transmembrane', 'LipidAnchor', 'Soluble']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/771 [00:00<?, ?B/s]

modeling_esm_plusplus.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Synthyra/ESMplusplus_large:
- modeling_esm_plusplus.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

In [ ]:
def embedding_extract(model, tokenizer, sequence, device):
    X_tokenized = tokenizer(
        sequence,
        padding = True,
        return_tensors ='pt',
        )
    X_tokenized = X_tokenized.to(device)
    with torch.no_grad():
        output = model(**X_tokenized, output_hidden_states=True)
    X_embeddings = output.hidden_states[-1].squeeze().cpu()
    return X_embeddings

In [ ]:
keys = []
df_partACC = df['PartACC'].values.tolist()
df_Y = df[Y_LABELS].values
df_sequence = df['Sequence'].values.tolist()

env = lmdb.open(
    SAVE_PATH,
    map_size=60*1024**3,
    writemap = True,
    map_async=True
    )

with env.begin(write=True) as txn:
    for i, partACC in tqdm.tqdm(enumerate(df_partACC), total = len(df_partACC)):
        sequences = df_sequence[i]
        y_targets = torch.FloatTensor(df_Y[i])
        X_embeddings = embedding_extract(MODEL_PRETRAINED, TOKENIZER, sequences, DEVICE)
        buffer = io.BytesIO()
        torch.save({'embedding' : X_embeddings, 'target' : y_targets}, buffer)
        txn.put(partACC.encode(), buffer.getvalue())
        keys.append(partACC)
    txn.put(b'__keys__', pickle.dumps(keys))

100%|██████████| 24801/24801 [28:26<00:00, 14.53it/s]
